# INCEpTION gold corpus — decision dossier (B)

This notebook is the methodology record for the fourteen normalisation decisions applied
by `scripts/gcn_gold/02_normalise.py`. For each one it measures the actual rows that made
the decision necessary, read fresh from `data/interim/gcn_gold_corpus/` only — not
imported from the script and not read from the corpus — so every figure here is one that
was visible at the moment the decision was taken. What each decision changed once applied
is measured in `C_normalisation.ipynb`; this notebook does not repeat that. It is written
for someone deciding whether to accept a decision, not someone checking that it was
applied.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 300)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/interim/gcn_gold_corpus").is_dir())
CORPUS = ROOT / "data/interim/gcn_gold_corpus"

TABLE_NAMES = ["documents", "annotators", "evidence_spans", "photometry_spans", "event_summaries"]
TABLES = {}
for name in TABLE_NAMES:
    path = CORPUS / f"{name}.parquet"
    frame = pd.read_parquet(path)
    if len(frame) == 0:
        raise ValueError(f"Table '{name}' loaded 0 rows from {path}")
    TABLES[name] = frame
    print(f"{name:18s} rows={len(frame):5d} columns={frame.shape[1]:4d}  <- {path}")

STRUCTURAL_SPAN_COLS = ["document_name", "layer_source", "xmi_id", "begin", "end", "covered_text"]
EVIDENCE_FEATURES = [c for c in TABLES["evidence_spans"].columns if c not in STRUCTURAL_SPAN_COLS]
PHOTOMETRY_FEATURES = [c for c in TABLES["photometry_spans"].columns if c not in STRUCTURAL_SPAN_COLS]

# decision 1: span_index, 0-based within (document_name, layer_source, begin, end)
for name in ["evidence_spans", "photometry_spans"]:
    TABLES[name] = TABLES[name].copy()
    TABLES[name]["span_index"] = TABLES[name].groupby(
        ["document_name", "layer_source", "begin", "end"]).cumcount()


def has_comment(value):
    """A comment counts as present when non-null and non-blank."""
    return value is not None and value.strip() != ""


def matched_pairs(df):
    """List of (baseline_index, annotator_index) matched on exact
    (document_name, begin, end, span_index) -- decision 2, reimplemented here independently."""
    baseline = df[df["layer_source"] == "INITIAL_CAS"]
    key_to_idx = {(r.document_name, r.begin, r.end, r.span_index): idx
                 for idx, r in baseline.iterrows()}
    pairs = []
    for idx, row in df[df["layer_source"] != "INITIAL_CAS"].iterrows():
        b_idx = key_to_idx.get((row.document_name, row.begin, row.end, row.span_index))
        if b_idx is not None:
            pairs.append((b_idx, idx))
    return pairs


print(f"\n{len(EVIDENCE_FEATURES)} evidence features: {EVIDENCE_FEATURES}")
print(f"{len(PHOTOMETRY_FEATURES)} photometry features: {PHOTOMETRY_FEATURES}")

documents          rows=   10 columns=   5  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/documents.parquet
annotators         rows=   28 columns=  10  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/annotators.parquet
evidence_spans     rows= 6610 columns=  12  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/evidence_spans.parquet
photometry_spans   rows= 2741 columns=  22  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/photometry_spans.parquet
event_summaries    rows=   28 columns=  26  <- /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_corpus/event_summaries.parquet

6 evidence features: ['label', 'target', 'certainty', 'value', 'unit', 'comment']
16 photometry features: ['measurement_type', 'photometric_system', 'target', 'certainty', 'magnitude_or_limit', 'magnitude_error', 'limit_sigma', 'unit', 'photometric_band', 'obs_time_raw', 'obs_time_type', 'obs_t

## Decisions 1 and 7 — what identifies a span

**Observed.** A photometry span can be identical on every feature between `INITIAL_CAS`
and an annotator's layer while carrying a different `xmi_id` on each side; separately,
7 offset ranges hold two distinct spans rather than one.
**Why it matters.** An id assigned inside one annotator's own file cannot be trusted to
name the same span in another file, and offsets alone are not always a unique key either.
**Decided.** A span is keyed by `(document_name, layer_source, begin, end, span_index)`.
`xmi_id` is retained as a column but never used to relate rows across layers.
**Scope.** Both span tables; 7 offset ranges in `photometry_spans` need `span_index` to
stay distinct.

In [2]:
ph = TABLES["photometry_spans"]
baseline = ph[ph["layer_source"] == "INITIAL_CAS"]
key_to_idx = {(r.document_name, r.begin, r.end, r.span_index): idx for idx, r in baseline.iterrows()}

example = None
for idx, row in ph[ph["layer_source"] != "INITIAL_CAS"].iterrows():
    b_idx = key_to_idx.get((row.document_name, row.begin, row.end, row.span_index))
    if b_idx is None:
        continue
    b_row = ph.loc[b_idx]
    if row["xmi_id"] != b_row["xmi_id"] and all(row[f] == b_row[f] for f in PHOTOMETRY_FEATURES):
        example = pd.DataFrame([b_row, row])
        break
print("a photometry span identical on every feature, xmi_id differing:")
cols = ["document_name", "layer_source", "xmi_id", "begin", "end", "covered_text"] + PHOTOMETRY_FEATURES
print(example[cols].to_string(index=False))

dup_keys = ph.groupby(["document_name", "layer_source", "begin", "end"]).size()
dup_keys = dup_keys[dup_keys > 1]
print(f"\n{len(dup_keys)} offset ranges carry two spans:")
print(dup_keys.reset_index(name="rows").to_string(index=False))

doc, ls, b, e = dup_keys.index[0]
group = ph[(ph["document_name"] == doc) & (ph["layer_source"] == ls)
          & (ph["begin"] == b) & (ph["end"] == e)]
differing = [c for c in PHOTOMETRY_FEATURES if group[c].nunique(dropna=False) > 1]
print(f"\none group in full ({doc} / {ls} [{b}:{e}]), differing features={differing}:")
print(group[["xmi_id", "span_index", "covered_text"] + differing].to_string(index=False))

a photometry span identical on every feature, xmi_id differing:
    document_name layer_source  xmi_id  begin  end                covered_text measurement_type photometric_system      target certainty magnitude_or_limit magnitude_error limit_sigma unit photometric_band          obs_time_raw       obs_time_type obs_time_reference exposure_time_raw timezone_raw instrument                                                                                                      comment
event_2025aji.xmi  INITIAL_CAS  159709   5599 5626 upper limit up to  20.7 mag      upper_limit            unknown counterpart confirmed               20.7                              mag                  240 sec after trigger relative_to_trigger    trigger_time_t0                                           Photometric band not found; the annotator must confirm it. Photometric system is unknown; verify AB or Vega.
event_2025aji.xmi      Camille  159669   5599 5626 upper limit up to  20.7 mag      upper_limit     

## Decision 2 — matching on exact offsets

**Observed.** 4,814 of 4,927 evidence annotator spans and 1,942 of 2,071 photometry
annotator spans match a baseline span on exact `(begin, end)`; every unmatched span on
either side was tested for interval overlap against the other side's unmatched spans, and
none overlaps.
**Why it matters.** A tolerance or overlap rule would have to decide how close is close
enough, and would risk pairing two spans that describe different things.
**Decided.** Match on exact `(begin, end, span_index)`. No tolerance, no overlap.
**Scope.** 6,998 annotator spans tested across both layers.

In [3]:
def unmatched(df):
    baseline = df[df["layer_source"] == "INITIAL_CAS"]
    annotator = df[df["layer_source"] != "INITIAL_CAS"]
    pairs = matched_pairs(df)
    matched_b = {b for b, a in pairs}
    matched_a = {a for b, a in pairs}
    return (baseline.loc[baseline.index.difference(matched_b)],
           annotator.loc[annotator.index.difference(matched_a)], len(baseline), len(annotator))


rows, sides = [], {}
for layer, df in [("evidence", TABLES["evidence_spans"]), ("photometry", TABLES["photometry_spans"])]:
    b_only, a_only, n_baseline, n_annotator = unmatched(df)
    sides[layer] = (b_only, a_only)
    rows.append({"layer": layer, "baseline_spans": n_baseline, "annotator_spans": n_annotator,
                 "matched_exact": n_annotator - len(a_only),
                 "baseline_unmatched": len(b_only), "annotator_unmatched": len(a_only)})
print(pd.DataFrame(rows).to_string(index=False))

print("\noverlap test on the spans exact matching left unmatched:")
for layer, (b_only, a_only) in sides.items():
    overlap_count = 0
    for doc in set(b_only["document_name"]) | set(a_only["document_name"]):
        bd, ad = b_only[b_only["document_name"] == doc], a_only[a_only["document_name"] == doc]
        for _, br in bd.iterrows():
            for _, ar in ad.iterrows():
                if br["begin"] < ar["end"] and ar["begin"] < br["end"]:
                    overlap_count += 1
    print(f"  {layer}: {len(b_only)} baseline-only, {len(a_only)} annotator-only; "
         f"pairs overlapping across the two sides = {overlap_count}")

     layer  baseline_spans  annotator_spans  matched_exact  baseline_unmatched  annotator_unmatched
  evidence            1683             4927           4814                   0                  113
photometry             670             2071           1942                   0                  129

overlap test on the spans exact matching left unmatched:
  evidence: 0 baseline-only, 113 annotator-only; pairs overlapping across the two sides = 0
  photometry: 0 baseline-only, 129 annotator-only; pairs overlapping across the two sides = 0


## Decisions 3 and 4 — what counts as a correction

**Observed.** Among matched pairs, some are identical on every feature, some differ on
one, and some differ on as many as eight; comparing the category feature alone
(`label` or `measurement_type`) misses most of what actually changed.
**Why it matters.** A rule that only asked "did the category change" would call the vast
majority of corrected spans unchanged.
**Decided.** A matched pair is `accepted` only if every feature is equal; otherwise it is
`corrected`, decided by comparing every feature column, not the category alone.
**Scope.** 6,756 matched pairs across both layers.

In [4]:
ph = TABLES["photometry_spans"]
pairs = matched_pairs(ph)
diffs = [(b, a, [f for f in PHOTOMETRY_FEATURES if ph.at[a, f] != ph.at[b, f]]) for b, a in pairs]

identical = next(d for d in diffs if len(d[2]) == 0)
one_diff = next(d for d in diffs if len(d[2]) == 1)
eight_diff = next(d for d in diffs if len(d[2]) == 8)
cols = ["document_name", "layer_source", "begin", "end"] + PHOTOMETRY_FEATURES
for label, (b, a, fields) in [("identical on every feature", identical),
                              ("differs on one feature", one_diff),
                              ("differs on eight features", eight_diff)]:
    print(f"\n{label} -- differing={fields}")
    print(ph.loc[[b, a], cols].to_string(index=False))

rows = []
for layer, df, features, cat in [("evidence", TABLES["evidence_spans"], EVIDENCE_FEATURES, "label"),
                                 ("photometry", TABLES["photometry_spans"], PHOTOMETRY_FEATURES,
                                  "measurement_type")]:
    layer_pairs = matched_pairs(df)
    category_differs = sum(1 for b, a in layer_pairs if df.at[a, cat] != df.at[b, cat])
    any_differs = sum(1 for b, a in layer_pairs
                      if any(df.at[a, f] != df.at[b, f] for f in features))
    rows.append({"layer": layer, "matched_pairs": len(layer_pairs),
                "category_alone_differs": category_differs, "any_feature_differs": any_differs})
print("\ncategory-alone vs any-feature comparison:")
print(pd.DataFrame(rows).to_string(index=False))


identical on every feature -- differing=[]
    document_name layer_source  begin  end measurement_type photometric_system      target certainty magnitude_or_limit magnitude_error limit_sigma unit photometric_band          obs_time_raw       obs_time_type obs_time_reference exposure_time_raw timezone_raw instrument                                                                                                      comment
event_2025aji.xmi  INITIAL_CAS   5599 5626      upper_limit            unknown counterpart confirmed               20.7                              mag                  240 sec after trigger relative_to_trigger    trigger_time_t0                                           Photometric band not found; the annotator must confirm it. Photometric system is unknown; verify AB or Vega.
event_2025aji.xmi      Camille   5599 5626      upper_limit            unknown counterpart confirmed               20.7                              mag                  240 sec after trigger 


category-alone vs any-feature comparison:
     layer  matched_pairs  category_alone_differs  any_feature_differs
  evidence           4814                       9                  224
photometry           1942                       2                  651


## Decision 5 — which fields change

**Observed.** The rate at which a feature differs between a matched pair varies from 0%
to about 30%, and the two most-changed features in photometry are free text
(`comment`, `instrument`), not the structured fields.
**Why it matters.** Knowing that spans were corrected is not the same as knowing what was
corrected; a per-feature breakdown is what makes the correction actionable.
**Decided.** Add `changed_fields`, the sorted list of feature names that differ from the
baseline, to every corrected row.
**Scope.** Every one of the 6 evidence and 16 photometry feature columns, over 6,756
matched pairs.

In [5]:
rows = []
for layer, df, features in [("evidence", TABLES["evidence_spans"], EVIDENCE_FEATURES),
                            ("photometry", TABLES["photometry_spans"], PHOTOMETRY_FEATURES)]:
    pairs = matched_pairs(df)
    for feature in features:
        differing = sum(1 for b, a in pairs if df.at[a, feature] != df.at[b, feature])
        rows.append({"layer": layer, "feature": feature, "matched_pairs": len(pairs),
                    "differing": differing, "pct": round(100 * differing / len(pairs), 1)})
feature_diff = pd.DataFrame(rows).sort_values("differing", ascending=False).reset_index(drop=True)
feature_diff

,layer,feature,matched_pairs,differing,pct
0,photometry,comment,1942,574,29.6
1,photometry,instrument,1942,213,11.0
2,evidence,comment,4814,211,4.4
3,photometry,photometric_system,1942,131,6.7
4,photometry,obs_time_reference,1942,124,6.4
5,photometry,obs_time_raw,1942,69,3.6
6,photometry,exposure_time_raw,1942,39,2.0
7,photometry,photometric_band,1942,34,1.8
8,photometry,obs_time_type,1942,33,1.7
9,evidence,certainty,4814,24,0.5


## Decision 6 — overlapping spans

**Observed.** 84 evidence and 10 photometry pairs overlap within the same
`(document_name, layer_source)`; a quarter of the evidence overlaps and one photometry
overlap already exist in `INITIAL_CAS`, and most evidence overlaps are genuine crossings,
not one span nested inside another.
**Why it matters.** Two overlapping spans can both be scientifically valid readings of the
same words under different categories; picking one would discard the other's evidence.
**Decided.** Retain every overlapping span. Resolve nothing. Flag `is_overlapping`.
**Scope.** 94 pairs across both layers.

In [6]:
def find_overlaps(df):
    pairs = []
    for _, group in df.groupby(["document_name", "layer_source"]):
        idx = group.index.tolist()
        for i in range(len(idx)):
            for j in range(i + 1, len(idx)):
                a, b = df.loc[idx[i]], df.loc[idx[j]]
                if a["begin"] < b["end"] and b["begin"] < a["end"]:
                    pairs.append((idx[i], idx[j]))
    return pairs


def shape(a, b):
    if a["begin"] == b["begin"] and a["end"] == b["end"]:
        return "identical"
    if a["begin"] <= b["begin"] and a["end"] >= b["end"]:
        return "a_contains_b"
    if b["begin"] <= a["begin"] and b["end"] >= a["end"]:
        return "b_contains_a"
    return "partial_cross"


ev = TABLES["evidence_spans"]
pairs = find_overlaps(ev)
by_source = pd.Series([ev.at[i, "layer_source"] for i, j in pairs]).value_counts()
print("overlapping evidence pairs by layer_source:")
print(by_source.to_string())
shapes = pd.Series([shape(ev.loc[i], ev.loc[j]) for i, j in pairs]).value_counts()
print("\ncontainment shape:")
print(shapes.to_string())

target_doc = "event_2025aji.xmi"
print(f"\nthe 'long GRB' / 'GRB 250129A' pair, across every layer of {target_doc}:")
for i, j in pairs:
    a, b = ev.loc[i], ev.loc[j]
    if a["document_name"] == target_doc and {a["label"], b["label"]} == \
            {"CLASSIFICATION_INTERPRETATION", "EVENT_IDENTITY"}:
        print(f"  layer_source={a['layer_source']}")
        print(f"    [{a['begin']}:{a['end']}] {a['label']}: {a['covered_text']!r}")
        print(f"    [{b['begin']}:{b['end']}] {b['label']}: {b['covered_text']!r}")

overlapping evidence pairs by layer_source:
INITIAL_CAS      21
Yodgor            9
Zhanat            8
Priyadarshini     8
Sarah             8
Dahlia            7
Camille           7
Patrice           6
Eslam             6
Xinyue            3
Andrii            1

containment shape:
partial_cross    81
b_contains_a      2
a_contains_b      1

the 'long GRB' / 'GRB 250129A' pair, across every layer of event_2025aji.xmi:
  layer_source=Camille
    [53691:53699] CLASSIFICATION_INTERPRETATION: 'long GRB'
    [53696:53707] EVENT_IDENTITY: 'GRB 250129A'
  layer_source=INITIAL_CAS
    [53691:53699] CLASSIFICATION_INTERPRETATION: 'long GRB'
    [53696:53707] EVENT_IDENTITY: 'GRB 250129A'
  layer_source=Patrice
    [53691:53699] CLASSIFICATION_INTERPRETATION: 'long GRB'
    [53696:53707] EVENT_IDENTITY: 'GRB 250129A'
  layer_source=Xinyue
    [53691:53699] CLASSIFICATION_INTERPRETATION: 'long GRB'
    [53696:53707] EVENT_IDENTITY: 'GRB 250129A'


## Decision 8 — the comment field

**Observed.** A matched pair's comment can be identical to the baseline, replaced with a
different comment, written where the baseline had none, or removed entirely; all four
occur, and "no comment on either side" is the single largest class.
**Why it matters.** Collapsing these into "comment changed: yes/no" would treat an
annotator deleting extractor guidance the same as an annotator adding a new note.
**Decided.** Add `comment_status` with four values; treat null and whitespace-only as no
comment for this classification, unlike decision 4/5's null-vs-empty-string distinction.
**Scope.** Every span row in both layers.

In [7]:
def classify_comment(b_comment, a_comment):
    b_has, a_has = has_comment(b_comment), has_comment(a_comment)
    if not b_has and not a_has:
        return "none"
    if b_has and a_has and b_comment == a_comment:
        return "identical"
    if b_has and a_has:
        return "replaced"
    if not b_has and a_has:
        return "written_where_none"
    return "removed"


examples, counts_by_layer = {}, []
for layer, df in [("evidence", TABLES["evidence_spans"]), ("photometry", TABLES["photometry_spans"])]:
    pairs = matched_pairs(df)
    classes = [classify_comment(df.at[b, "comment"], df.at[a, "comment"]) for b, a in pairs]
    counts_by_layer.append({"layer": layer, **pd.Series(classes).value_counts().to_dict()})
    for cls, (b, a) in zip(classes, pairs):
        examples.setdefault(cls, (layer, b, a))

for cls in ["identical", "replaced", "written_where_none", "removed"]:
    layer, b, a = examples[cls]
    df = TABLES["evidence_spans"] if layer == "evidence" else TABLES["photometry_spans"]
    print(f"\n{cls} ({layer}):")
    print(f"  baseline:   {df.at[b, 'comment']!r}")
    print(f"  annotator:  {df.at[a, 'comment']!r}")

print("\ncounts per class per layer:")
print(pd.DataFrame(counts_by_layer).fillna(0).astype({c: int for c in
     ["identical", "replaced", "written_where_none", "removed", "none"]}).to_string(index=False))


identical (evidence):
  baseline:   'Day-fraction format (MASTER/Fermi); it may correspond to a GRB with an official letter suffix. Verify the mapping to the canonical event.'
  annotator:  'Day-fraction format (MASTER/Fermi); it may correspond to a GRB with an official letter suffix. Verify the mapping to the canonical event.'

replaced (evidence):
  baseline:   'Redshift without explicit attribution to event or context; verify whether it belongs to the event/afterglow/host or to a context galaxy.'
  annotator:  'redshift context'

written_where_none (evidence):
  baseline:   ''
  annotator:  'Added units of J2000 degrees'

removed (evidence):
  baseline:   'Trigger time without an adjacent date in the text; the annotator must complete the date (for example, from the event name or context).'
  annotator:  None

counts per class per layer:
     layer  none  identical  replaced  written_where_none  removed
  evidence  3837        767       133                  61       16
photometry   

## Decisions 9 and 10 — spans without a category

**Observed.** 18 evidence spans carry no `label` and 44 photometry spans no
`measurement_type`; none of these are in `INITIAL_CAS`. Most populate only a comment and
nothing else, but a handful are fully worked measurements missing only their category.
**Why it matters.** A span with no category still records that a human marked that text;
dropping it would discard both the working notes and the near-complete measurements.
**Decided.** Retain every span without a category. Add `has_category`. Flag
`is_annotator_note` for created, categoryless spans that carry a comment.
**Scope.** 62 rows across both layers.

In [8]:
for layer, df, cat in [("evidence", TABLES["evidence_spans"], "label"),
                       ("photometry", TABLES["photometry_spans"], "measurement_type")]:
    no_cat = df[df[cat].isna()]
    features = EVIDENCE_FEATURES if layer == "evidence" else PHOTOMETRY_FEATURES
    other = [f for f in features if f != cat]
    print(f"\n{layer}: {len(no_cat)} spans with null {cat}")
    print(no_cat["layer_source"].value_counts().to_string())
    pop_count = no_cat[other].notna().sum(axis=1) - (no_cat[other] == "").sum(axis=1)
    print(f"how many of the other {len(other)} features are populated:")
    print(pop_count.value_counts().sort_index().to_string())

print("\nfive spans that carry a comment and little else:")
ev = TABLES["evidence_spans"]
sparse = ev[ev["label"].isna() & ev["comment"].map(has_comment)]
cols = ["document_name", "layer_source", "begin", "end", "covered_text", "target", "comment"]
print(sparse[cols].head(5).to_string(index=False))

print("\nfully-worked measurements missing only their category:")
ph = TABLES["photometry_spans"]
worked = ph[ph["measurement_type"].isna()]
other = [f for f in PHOTOMETRY_FEATURES if f != "measurement_type"]
pop_count = worked[other].notna().sum(axis=1) - (worked[other] == "").sum(axis=1)
print(worked.loc[pop_count >= 10, ["document_name", "layer_source", "begin", "end", "covered_text"]
                 + other].to_string(index=False))


evidence: 18 spans with null label
layer_source
Sarah      12
Camille     3
Patrice     2
Xinyue      1
how many of the other 5 features are populated:
0     3
1    11
3     3
4     1

photometry: 44 spans with null measurement_type
layer_source
Sarah      41
Camille     3
how many of the other 15 features are populated:
1      4
2     34
3      2
12     3
13     1

five spans that carry a comment and little else:
              document_name layer_source  begin   end                                               covered_text target                                                                                                 comment
          event_2026owq.xmi        Sarah  52089 52096                                                    GRANDMA   None                                                                          gcn to be excluded car grandma
          event_2026owq.xmi        Sarah  57259 57317 https://heaiki.ru/lvc/r/GRBs/GRB260610B/GRB260610B_LC1.jpg   None              

## Decision 11 — values that are not numbers

**Observed.** `exposure_time_raw` fails to parse as a float on roughly half its populated
values, because it holds text like `'5s'` and `'4x90s exposures'`, not a bare number;
the other three numeric-looking features fail rarely.
**Why it matters.** A companion numeric column that silently drops unparseable text would
make "no value" and "a value that could not be typed" look identical.
**Decided.** Add a `_numeric` companion column per feature holding the parsed float, or
null where it does not parse; the original column is retained unchanged.
**Scope.** 4 photometry features.

In [9]:
def parse_float(value):
    if value is None or value == "":
        return None
    try:
        return float(value)
    except ValueError:
        return None


ph = TABLES["photometry_spans"]
for feature in ["magnitude_or_limit", "magnitude_error", "limit_sigma", "exposure_time_raw"]:
    populated = ph[feature][ph[feature].notna() & (ph[feature] != "")]
    parsed = populated.map(parse_float)
    failed = populated[parsed.isna()]
    pct = 100 * len(failed) / len(populated)
    print(f"{feature}: {len(failed)} of {len(populated)} populated values fail to parse "
         f"({pct:.1f}%)")
    if feature == "exposure_time_raw":
        print("  (roughly half its values are not bare numbers)")
    print(f"  failing values: {sorted(failed.unique().tolist())}")

magnitude_or_limit: 1 of 2698 populated values fail to parse (0.0%)
  failing values: ['22.04 | 20.07']
magnitude_error: 3 of 1723 populated values fail to parse (0.2%)
  failing values: ['0.35 | 0.17', 'n/d', 'unknown']
limit_sigma: 2 of 48 populated values fail to parse (4.2%)
  failing values: ['18.3 mag', 'r']
exposure_time_raw: 915 of 1867 populated values fail to parse (49.0%)
  (roughly half its values are not bare numbers)
  failing values: ['10*300 Rc', '100 sec', '100*30', '100sx20', '100x20s', '100x80', '103 min', '103*60', '103min', '105x60s', '10s', '10x100s', '10x180s', '10x240s', '10x300', '10x600s', '10x90s exposures', '11 x 180s', '116*90 R', '11min', '11x60s', '12 x 300 sec', '12 x 300s', '120x60', '12x300', '12x300s', '13*300 Rc', '14x180s ', '15 s', '15 x 180s', '15 x 600', '15*180', '15*60', '150 s', '150s', '150x80', '15x600', '16*100 s', '16x300s', '1980s', '2 exposures of 600 s', '2*120 R', '2*300', '20 x 180 s ', '20*100 s', '20*60', '200s', '200s*8', '200sx8',

## Decisions 12 and 14 — vocabulary

**Observed.** Every value an annotator used for a categorical feature already appears
somewhere in `INITIAL_CAS`, except in photometry, where `certainty`, `obs_time_type` and
`obs_time_reference` each carry a value the extractor never emits; free-text features carry
hundreds of distinct values with no repeated closed vocabulary behind them.
**Why it matters.** A mapping rule can only be written against a vocabulary that is
closed; free text with hundreds of distinct hand-written values is not.
**Decided.** Flag `extractor_vocabulary_gap` on categorical features only. Normalise no
free-text value: no mapping, no trimming, no recoding.
**Scope.** 6 categorical and 6 free-text features checked.

In [10]:
categorical = {"evidence_spans": ["label", "target", "certainty"],
              "photometry_spans": ["measurement_type", "photometric_system", "target",
                                   "certainty", "obs_time_type", "obs_time_reference"]}
for table_name, features in categorical.items():
    df = TABLES[table_name]
    baseline, annotator = df[df["layer_source"] == "INITIAL_CAS"], df[df["layer_source"] != "INITIAL_CAS"]
    for f in features:
        b_vals, a_vals = set(baseline[f].dropna()), set(annotator[f].dropna())
        gap = sorted(a_vals - b_vals)
        print(f"{table_name}.{f}: baseline has {len(b_vals)} values, annotators use "
             f"{len(a_vals)}; used by annotators but never in baseline: {gap}")

free_text = {"evidence_spans": ["value", "unit", "comment"],
            "photometry_spans": ["magnitude_or_limit", "magnitude_error", "photometric_band",
                                 "obs_time_raw", "exposure_time_raw", "timezone_raw", "instrument"]}
print()
for table_name, features in free_text.items():
    df = TABLES[table_name]
    for f in features:
        populated = df[f][df[f].notna() & (df[f] != "")]
        print(f"{table_name}.{f}: {populated.nunique()} distinct populated values "
             f"(no closed vocabulary)")

evidence_spans.label: baseline has 15 values, annotators use 15; used by annotators but never in baseline: []
evidence_spans.target: baseline has 5 values, annotators use 5; used by annotators but never in baseline: []
evidence_spans.certainty: baseline has 5 values, annotators use 5; used by annotators but never in baseline: []
photometry_spans.measurement_type: baseline has 2 values, annotators use 2; used by annotators but never in baseline: []
photometry_spans.photometric_system: baseline has 3 values, annotators use 3; used by annotators but never in baseline: []
photometry_spans.target: baseline has 2 values, annotators use 2; used by annotators but never in baseline: []
photometry_spans.certainty: baseline has 2 values, annotators use 3; used by annotators but never in baseline: ['candidate']
photometry_spans.obs_time_type: baseline has 5 values, annotators use 7; used by annotators but never in baseline: ['calendar_date', 'start_time_plus_exposure']
photometry_spans.obs_time_re

## Decision 13 — no consensus layer

**Observed.** Every document carries between two and four annotators, each producing an
independent full pass over the text; at shared offsets within one document, annotators
sometimes agree on a label and sometimes do not.
**Why it matters.** The project defines a curation workflow but never ran it — there is no
merged answer to read instead of the individual layers.
**Decided.** Retain `INITIAL_CAS` as its own layer. Merge nothing. Resolve no disagreement
between annotators.
**Scope.** All 9,351 span rows.

In [11]:
grid_rows = []
for doc in sorted(TABLES["documents"]["document_name"]):
    row = {"document": doc}
    for layer_key, df in [("evidence", TABLES["evidence_spans"]), ("photometry", TABLES["photometry_spans"])]:
        counts = df[df["document_name"] == doc].groupby("layer_source").size()
        row[f"{layer_key}_baseline"] = int(counts.get("INITIAL_CAS", 0))
        annotators = sorted(a for a in counts.index if a != "INITIAL_CAS")
        row[f"{layer_key}_annotators"] = ", ".join(f"{a}={counts[a]}" for a in annotators)
    grid_rows.append(row)
grid = pd.DataFrame(grid_rows)
print(grid.to_string(index=False))

doc = "event_2025aji.xmi"
ev = TABLES["evidence_spans"]
doc_df = ev[ev["document_name"] == doc]
pivot = doc_df.pivot_table(index=["begin", "end"], columns="layer_source", values="label",
                          aggfunc="first")
print(f"\nlabel at every shared offset in {doc}, one column per layer_source:")
pivot

                   document  evidence_baseline                                   evidence_annotators  photometry_baseline                             photometry_annotators
          event_2025aji.xmi                201                  Camille=200, Patrice=202, Xinyue=202                  109              Camille=122, Patrice=109, Xinyue=109
          event_2026owq.xmi                204                     Dahlia=204, Sarah=209, Zhanat=205                   68                    Dahlia=69, Sarah=79, Zhanat=69
 event_EP-260623_025405.xmi                 96                                 Dahlia=106, Yodgor=96                   19                              Dahlia=24, Yodgor=25
event_GCN-251013_173943.xmi                237                      Andrii=237, Eslam=237, Sarah=253                  266                  Andrii=266, Eslam=266, Sarah=290
event_GCN-251222_170549.xmi                242                        Camille=242, Priyadarshini=256                   37                   

,layer_source,Camille,INITIAL_CAS,Patrice,Xinyue
begin,end,,,,
45,56,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY
1133,1145,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY
1382,1404,LOCALIZATION,LOCALIZATION,LOCALIZATION,LOCALIZATION
1801,1812,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY
1849,1868,COUNTERPART_ASSOCIATION,COUNTERPART_ASSOCIATION,COUNTERPART_ASSOCIATION,COUNTERPART_ASSOCIATION
2106,2117,TRIGGER_TIME,TRIGGER_TIME,TRIGGER_TIME,TRIGGER_TIME
2123,2150,TRIGGER_INSTRUMENT,TRIGGER_INSTRUMENT,TRIGGER_INSTRUMENT,TRIGGER_INSTRUMENT
2179,2190,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY,EVENT_IDENTITY
2291,2314,LOCALIZATION,NaN,NaN,NaN


## What this record establishes

Each of the fourteen decisions appears here with the data that motivated
it, measured independently of the code that applies it. None was taken by
convention: every one answers to something observable in the flattened
tables.

Three kinds of justification run through this record. Some decisions
record what an annotator did and the schema does not capture: the five
spans Camille did not carry forward, and the 47 created without a
category solely to leave an observation. Others make explicit what was
implicit: `match_status` distinguishes four cases that the export itself
only allows to be inferred by comparison against the baseline. And one,
decision 13, declines to resolve what is not resolved: without a curation
layer, the corpus holds 28 independent validations rather than one answer
per document.

Two decisions consist of not intervening. Overlapping spans are kept
because the domain produces them — `'long GRB'` and `'GRB 250129A'` share
the word GRB, and three annotators kept that pair as it stood — and free
text is left as written, because ten people annotating by hand do not
produce a closed vocabulary.